# Exercise 62 — Date and time handling

## Concept

Dates come into pandas as strings 99% of the time. Convert once, work with `datetime64[ns]` after that.

```python
df["ts"] = pd.to_datetime(df["ts"])                          # auto-detect
df["ts"] = pd.to_datetime(df["ts"], format="%Y-%m-%d %H:%M")  # explicit (faster, safer)
df["ts"] = pd.to_datetime(df["ts"], errors="coerce")          # bad strings → NaT
```

### `.dt` accessor — extract components

```python
df["ts"].dt.year
df["ts"].dt.month
df["ts"].dt.day
df["ts"].dt.hour
df["ts"].dt.day_name()           # 'Monday', ...
df["ts"].dt.date                 # just the date part (Python date objects)
df["ts"].dt.to_period("M")       # month period: '2026-01'
```

### Arithmetic

`pd.Timedelta` for durations:

```python
df["ts"] + pd.Timedelta(days=1)
(df["end"] - df["start"]).dt.total_seconds()
```

### Resampling time series

With a datetime index (or `on=` parameter), `.resample("D")` / `"W"` / `"M"` aggregates over time buckets — like `groupby` but for time:

```python
df.set_index("ts").resample("D")["amount"].sum()
```

Resample frequencies: `"D"` day, `"W"` week, `"ME"` month-end, `"H"` hour, etc.

### Time zones (briefly)

Naive timestamps have no tz. Localize once at the boundary:

```python
ts.dt.tz_localize("UTC").dt.tz_convert("America/Sao_Paulo")
```

Unless you're explicitly working in multiple timezones, keep everything in UTC internally and only convert at the display layer.

## Setup

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "ts":       [
        "2026-01-01 09:00",
        "2026-01-01 14:30",
        "2026-01-02 10:00",
        "2026-01-03 16:45",
        "2026-01-03 18:00",
        "2026-01-05 08:15",
        "2026-01-05 22:00",
        "2026-01-07 12:00",
    ],
    "amount":   [10, 25, 12, 40, 8, 15, 30, 22],
})
print(df)


## Task 1 — Parse and add components

Return a new DataFrame where:
- `ts` is converted to `datetime64[ns]` (no longer string)
- A new column `date` holds the date portion (as a Python `date`)
- A new column `hour` holds the hour (integer 0–23)
- A new column `day_name` holds the English day name (e.g., `"Monday"`)

```python
def with_date_parts(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Don't mutate the input.

In [ ]:
import pandas as pd
from datetime import date

def with_date_parts(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = with_date_parts(df)
assert result["ts"].dtype.kind == "M"        # datetime
assert result.loc[0, "date"] == date(2026, 1, 1)
assert result.loc[1, "hour"] == 14
assert result.loc[0, "day_name"] == "Thursday"   # 2026-01-01 is a Thursday
assert df["ts"].dtype == object                  # input unchanged
print("ok")


## Task 2 — Daily totals via resample

Return a DataFrame with columns `["date", "total"]` — one row per calendar day, summing `amount`. Days with no orders should be present with `total = 0` (so days like 2026-01-04 and 2026-01-06 appear too).

```python
def daily_totals(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Hint: convert `ts`, set it as the index, `.resample("D")["amount"].sum()`. Then `.reset_index()` to get the index back as a column, and `.rename({"ts": "date"})` to tidy.

Expected: 7 rows from 2026-01-01 through 2026-01-07; 2026-01-04 and 2026-01-06 have total 0.

In [ ]:
import pandas as pd

def daily_totals(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = daily_totals(df)
assert list(result.columns) == ["date", "total"]
assert len(result) == 7
assert result["total"].sum() == df["amount"].sum()
# Days with no orders → 0
row_04 = result[result["date"] == pd.Timestamp("2026-01-04")].iloc[0]
assert row_04["total"] == 0
print("ok")


## Task 3 — Filter by date range

Return rows whose `ts` falls within `[start, end]` (both inclusive, as date strings like `"2026-01-02"` / `"2026-01-05"`).

```python
def rows_between(df: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    ...
```

Hint: convert the bounds with `pd.to_datetime(...)` and the `ts` column too. The bounds at midnight mean "end" inclusive at the start of that day. To make `end` truly inclusive of the entire day, you can compare `ts.dt.date <= pd.to_datetime(end).date()`.

Expected: `rows_between(df, "2026-01-02", "2026-01-03")` → order_ids `{3, 4, 5}`.

In [ ]:
import pandas as pd

def rows_between(df: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    pass  # TODO

result = rows_between(df, "2026-01-02", "2026-01-03")
assert set(result["order_id"]) == {3, 4, 5}
print("ok")


## Task 4 — Duration between two timestamps

Given a DataFrame with `start_ts` and `end_ts` columns (as strings), return a new DataFrame with a column `duration_minutes` (float) computed as `(end_ts - start_ts).total_seconds() / 60`.

```python
def compute_duration(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Use `pd.to_datetime` on both columns, then arithmetic. The result of subtracting two datetime Series is a `timedelta64[ns]` Series; the `.dt.total_seconds()` accessor gives float seconds.

In [ ]:
import pandas as pd

spans = pd.DataFrame({
    "job":       ["a", "b", "c"],
    "start_ts":  ["2026-01-01 10:00:00", "2026-01-01 10:00:00", "2026-01-02 23:30:00"],
    "end_ts":    ["2026-01-01 10:30:00", "2026-01-01 12:15:00", "2026-01-03 00:00:00"],
})

def compute_duration(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = compute_duration(spans)
assert result["duration_minutes"].tolist() == [30.0, 135.0, 30.0]
print("ok")
